In [1]:
#libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# using currrent path
project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

raw_path = project_path / "data" / "raw"
processed_path = project_path / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)

print("Raw folder:", raw_path)
print("Processed folder:", processed_path)

Raw folder: c:\Users\naren\OneDrive\Desktop\Data Analytics\Projects\Portfolio Centerpiece Projects using AI\End-to-End Amazon Prime Customer Churn & Retention Analytics\data\raw
Processed folder: c:\Users\naren\OneDrive\Desktop\Data Analytics\Projects\Portfolio Centerpiece Projects using AI\End-to-End Amazon Prime Customer Churn & Retention Analytics\data\processed


In [3]:
# Loading all raw files
customers = pd.read_csv(raw_path / "customers.csv", low_memory=False)
memberships = pd.read_csv(raw_path / "memberships.csv", low_memory=False)
orders = pd.read_csv(raw_path / "orders.csv", low_memory=False)
prime_video = pd.read_csv(raw_path / "prime_video_activity.csv", low_memory=False)
payments = pd.read_csv(raw_path / "payments.csv", low_memory=False)
support = pd.read_csv(raw_path / "support_interactions.csv", low_memory=False)

print("All files loaded")

All files loaded


In [4]:
# Checking the structure of every file
dataframes = {
    "customers": customers,
    "memberships": memberships,
    "orders": orders,
    "prime_video_activity": prime_video,
    "payments": payments,
    "support_interactions": support
}

primary_keys = {
    "customers": "customer_id",
    "memberships": "membership_id",
    "orders": "order_id",
    "prime_video_activity": "activity_id",
    "payments": "payment_id",
    "support_interactions": "ticket_id"
}

for name, df in dataframes.items():
    print("\n", name.upper())
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Column names:", df.columns.tolist())
    print("Memory MB:", round(df.memory_usage(deep=True).sum() / 1024**2, 2))
    print("Primary key unique:", df[primary_keys[name]].is_unique)
    print(df.dtypes)
    print(df.head(3))


 CUSTOMERS
Rows: 50050
Columns: 11
Column names: ['customer_id', 'signup_date', 'country', 'state', 'city_tier', 'age_group', 'preferred_language', 'acquisition_channel', 'primary_device', 'email_marketing_opt_in', 'account_status']
Memory MB: 29.78
Primary key unique: False
customer_id               object
signup_date               object
country                   object
state                     object
city_tier                 object
age_group                 object
preferred_language        object
acquisition_channel       object
primary_device            object
email_marketing_opt_in    object
account_status            object
dtype: object
  customer_id signup_date        country        state city_tier age_group  \
0  CUST025026  2025-09-22          India  West Bengal    Tier 1     45-54   
1  CUST041150  2025-11-14          Japan     Kanagawa     Metro       65+   
2  CUST020499  2024-09-29  United States        Texas    Tier 1     55-64   

  preferred_language acquisition_chan

In [5]:
# Checking data quality before cleaning
print("Missing customer IDs")
print("customers:", customers["customer_id"].isna().sum())
print("memberships:", memberships["customer_id"].isna().sum())
print("orders:", orders["customer_id"].isna().sum())
print("prime video:", prime_video["customer_id"].isna().sum())
print("payments:", payments["customer_id"].isna().sum())
print("support:", support["customer_id"].isna().sum())

print("\nDuplicate primary keys")
for name, df in dataframes.items():
    key = primary_keys[name]
    print(name, ":", df[key].duplicated().sum())

customer_ids = set(customers["customer_id"].dropna())
membership_ids = set(memberships["membership_id"].dropna())

print("\nOrphan foreign keys")
print("memberships:", (~memberships["customer_id"].isin(customer_ids)).sum())
print("orders:", (~orders["customer_id"].isin(customer_ids)).sum())
print("prime video:", (~prime_video["customer_id"].isin(customer_ids)).sum())
print("payments customer:", (~payments["customer_id"].isin(customer_ids)).sum())
print("payments membership:", (~payments["membership_id"].isin(membership_ids)).sum())
print("support:", (~support["customer_id"].isin(customer_ids)).sum())

print("\nInvalid numeric values")
print("Negative order values:", (orders["order_value"] < 0).sum())
print("Negative watch minutes:", (prime_video["watch_minutes"] < 0).sum())
print("Invalid payment amounts:", ((payments["payment_amount"] < 0) | payments["payment_amount"].isna()).sum())
print("Invalid satisfaction scores:", ((support["satisfaction_score"].notna()) & (~support["satisfaction_score"].between(1, 5))).sum())

signup_check = pd.to_datetime(customers["signup_date"], errors="coerce")
start_check = pd.to_datetime(memberships["membership_start_date"], errors="coerce")
end_check = pd.to_datetime(memberships["membership_end_date"], errors="coerce")
cancel_check = pd.to_datetime(memberships["cancellation_date"], errors="coerce")
order_date_check = pd.to_datetime(orders["order_date"], errors="coerce")
video_date_check = pd.to_datetime(prime_video["activity_date"], errors="coerce")
payment_date_check = pd.to_datetime(payments["payment_date"], errors="coerce")
ticket_date_check = pd.to_datetime(support["ticket_date"], errors="coerce")

print("\nInvalid dates")
print("signup dates:", (customers["signup_date"].notna() & signup_check.isna()).sum())
print("membership start dates:", (memberships["membership_start_date"].notna() & start_check.isna()).sum())
print("membership end dates:", (memberships["membership_end_date"].notna() & end_check.isna()).sum())
print("cancellation dates:", (memberships["cancellation_date"].notna() & cancel_check.isna()).sum())
print("order dates:", (orders["order_date"].notna() & order_date_check.isna()).sum())
print("video dates:", (prime_video["activity_date"].notna() & video_date_check.isna()).sum())
print("payment dates:", (payments["payment_date"].notna() & payment_date_check.isna()).sum())
print("ticket dates:", (support["ticket_date"].notna() & ticket_date_check.isna()).sum())

print("\nInvalid membership dates")
print("End before start:", ((end_check.notna()) & (end_check < start_check)).sum())
print("Cancellation before start:", ((cancel_check.notna()) & (cancel_check < start_check)).sum())

signup_lookup = customers.assign(signup_check=signup_check).drop_duplicates("customer_id").set_index("customer_id")["signup_check"]
print("Orders before signup:", (order_date_check < orders["customer_id"].map(signup_lookup)).sum())

membership_check = memberships.copy()
membership_check["start_check"] = start_check
membership_check["cancel_check"] = cancel_check
membership_check["status_check"] = membership_check["membership_status"].astype("string").str.strip().replace({
    "Canceled": "Cancelled",
    "active": "Active",
    "expired": "Expired",
    "paused": "Paused",
    "payment pending": "Payment Pending"
})
latest_check = membership_check.sort_values(["customer_id", "start_check"]).drop_duplicates("customer_id", keep="last")
cancel_lookup = latest_check[latest_check["status_check"].eq("Cancelled")].set_index("customer_id")["cancel_check"]

print("Orders after latest cancellation:", (order_date_check > orders["customer_id"].map(cancel_lookup)).sum())
print("Video activity after latest cancellation:", (video_date_check > prime_video["customer_id"].map(cancel_lookup)).sum())

valid_membership_status = ["Active", "Cancelled", "Expired", "Paused", "Payment Pending"]
valid_payment_status = ["Successful", "Failed", "Pending", "Refunded"]

raw_membership_status = memberships["membership_status"].astype("string").str.strip()
raw_payment_status = payments["payment_status"].astype("string").str.strip()

print("Invalid membership status values:", (~raw_membership_status.isin(valid_membership_status)).sum())
print("Invalid payment status values:", (~raw_payment_status.isin(valid_payment_status)).sum())

Missing customer IDs
customers: 0
memberships: 0
orders: 0
prime video: 0
payments: 0
support: 0

Duplicate primary keys
customers : 50
memberships : 55
orders : 400
prime_video_activity : 550
payments : 125
support_interactions : 35

Orphan foreign keys
memberships: 0
orders: 0
prime video: 0
payments customer: 0
payments membership: 0
support: 0

Invalid numeric values
Negative order values: 0
Negative watch minutes: 0
Invalid payment amounts: 0
Invalid satisfaction scores: 0

Invalid dates
signup dates: 12
membership start dates: 0
membership end dates: 4
cancellation dates: 4
order dates: 24
video dates: 30
payment dates: 16
ticket dates: 10

Invalid membership dates
End before start: 0
Cancellation before start: 0
Orders before signup: 0
Orders after latest cancellation: 0
Video activity after latest cancellation: 0
Invalid membership status values: 220
Invalid payment status values: 500


In [6]:
# Removing exact duplicate rows
customers = customers.drop_duplicates().copy()
memberships = memberships.drop_duplicates().copy()
orders = orders.drop_duplicates().copy()
prime_video = prime_video.drop_duplicates().copy()
payments = payments.drop_duplicates().copy()
support = support.drop_duplicates().copy()

print("Exact duplicates removed")

Exact duplicates removed


In [7]:
# Removing extra spaces from text columns
for df in [customers, memberships, orders, prime_video, payments, support]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("string").str.strip()

# Standardise customer values
customers["country"] = customers["country"].str.title().replace({
    "Usa": "United States",
    "Uk": "United Kingdom"
})

customers["acquisition_channel"] = customers["acquisition_channel"].replace({
    "Paid Ads": "Paid Advertising",
    "Device promo": "Device Promotion",
    "Student Promo": "Student Promotion",
    "amazon shopping": "Amazon Shopping",
    "organic search": "Organic Search",
    "Social media": "Social Media",
    "referral": "Referral"
})

device_map = {
    "Fire tv": "Fire TV",
    "SmartTV": "Smart TV",
    "desktop": "Desktop",
    "game console": "Game Console",
    "mobile": "Mobile",
    "tablet": "Tablet"
}

customers["primary_device"] = customers["primary_device"].replace(device_map)

# Standardise membership values
memberships["plan_type"] = memberships["plan_type"].replace({
    "prime standard": "Prime Standard",
    "Prime student": "Prime Student",
    "Prime trial": "Prime Trial"
})

memberships["membership_status"] = memberships["membership_status"].replace({
    "Canceled": "Cancelled",
    "active": "Active",
    "expired": "Expired",
    "paused": "Paused",
    "payment pending": "Payment Pending"
})

# Standardise order values
orders["product_category"] = orders["product_category"].replace({
    "electronics": "Electronics",
    "grocery": "Grocery",
    "Health and Personal Care": "Health & Personal Care",
    "Home and Kitchen": "Home & Kitchen",
    "Sports and Outdoors": "Sports & Outdoors",
    "Toys and Games": "Toys & Games"
})

orders["delivery_speed"] = orders["delivery_speed"].replace({
    "2-Day": "Two-Day",
    "Same Day": "Same-Day",
    "one-day": "One-Day",
    "standard": "Standard"
})

# Standardise Prime Video values
prime_video["content_type"] = prime_video["content_type"].replace({
    "Live Sport": "Live Sports",
    "TV series": "TV Series",
    "documentary": "Documentary",
    "kids": "Kids",
    "movie": "Movie"
})

prime_video["device_type"] = prime_video["device_type"].replace(device_map)

# Standardise payment values
payments["payment_method"] = payments["payment_method"].replace({
    "Bank Account": "Bank Transfer",
    "Bank transfer": "Bank Transfer",
    "Debit card": "Debit Card",
    "Gift balance": "Gift Balance",
    "Wallet": "Digital Wallet",
    "credit card": "Credit Card",
    "upi": "UPI"
})

payments["payment_status"] = payments["payment_status"].replace({
    "failed": "Failed",
    "refunded": "Refunded",
    "successful": "Successful"
})

# Standardizing support values
support["issue_category"] = support["issue_category"].replace({
    "Account Login": "Account Access",
    "Cancel Request": "Cancellation Request",
    "Delivery issue": "Delivery Problem",
    "General query": "General Question",
    "membership billing": "Membership Billing",
    "payment failure": "Payment Failure",
    "Prime video issue": "Prime Video Issue",
    "refund request": "Refund Request"
})

support["support_channel"] = support["support_channel"].replace({
    "Help centre": "Help Center",
    "chat": "Chat",
    "e-mail": "Email",
    "phone": "Phone",
    "social media": "Social Media"
})

print("Text values standardised")

Text values standardised


In [8]:
# Converting date columns
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")

for col in ["membership_start_date", "renewal_date", "membership_end_date", "cancellation_date"]:
    memberships[col] = pd.to_datetime(memberships[col], errors="coerce")

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
prime_video["activity_date"] = pd.to_datetime(prime_video["activity_date"], errors="coerce")
payments["payment_date"] = pd.to_datetime(payments["payment_date"], errors="coerce")
support["ticket_date"] = pd.to_datetime(support["ticket_date"], errors="coerce")

# Repair missing signup dates using the first related activity
first_dates = pd.concat([
    memberships.groupby("customer_id")["membership_start_date"].min().rename("membership"),
    orders.groupby("customer_id")["order_date"].min().rename("order"),
    prime_video.groupby("customer_id")["activity_date"].min().rename("video"),
    payments.groupby("customer_id")["payment_date"].min().rename("payment"),
    support.groupby("customer_id")["ticket_date"].min().rename("support")
], axis=1).min(axis=1)

missing_signup = customers["signup_date"].isna()
customers.loc[missing_signup, "signup_date"] = customers.loc[missing_signup, "customer_id"].map(first_dates)

# Repair membership dates only when another valid date is available
ended_status = memberships["membership_status"].isin(["Cancelled", "Expired"])
cancelled_status = memberships["membership_status"].eq("Cancelled")

memberships.loc[ended_status & memberships["membership_end_date"].isna(), "membership_end_date"] = (
    memberships["cancellation_date"].fillna(memberships["renewal_date"])
)

memberships.loc[cancelled_status & memberships["cancellation_date"].isna(), "cancellation_date"] = (
    memberships["membership_end_date"].fillna(memberships["renewal_date"])
)

memberships.loc[ended_status & memberships["renewal_date"].isna(), "renewal_date"] = (
    memberships["membership_end_date"].fillna(memberships["cancellation_date"])
)

print("Dates converted and recoverable dates repaired")

Dates converted and recoverable dates repaired


In [9]:
# Convertig Boolean columns
boolean_map = {"Yes": True, "No": False}

customers["email_marketing_opt_in"] = customers["email_marketing_opt_in"].map(boolean_map).astype("boolean")
memberships["auto_renew_enabled"] = memberships["auto_renew_enabled"].map(boolean_map).astype("boolean")
orders["delivered_late_flag"] = orders["delivered_late_flag"].map(boolean_map).astype("boolean")
orders["returned_flag"] = orders["returned_flag"].map(boolean_map).astype("boolean")
payments["refund_flag"] = payments["refund_flag"].map(boolean_map).astype("boolean")
support["resolved_flag"] = support["resolved_flag"].map(boolean_map).astype("boolean")
support["repeat_contact_flag"] = support["repeat_contact_flag"].map(boolean_map).astype("boolean")

# Convert numeric columns
orders["order_value"] = pd.to_numeric(orders["order_value"], errors="coerce")
orders["items_count"] = pd.to_numeric(orders["items_count"], errors="coerce")
orders["shipping_fee_saved"] = pd.to_numeric(orders["shipping_fee_saved"], errors="coerce")

prime_video["watch_minutes"] = pd.to_numeric(prime_video["watch_minutes"], errors="coerce")
prime_video["sessions_count"] = pd.to_numeric(prime_video["sessions_count"], errors="coerce")
prime_video["titles_watched"] = pd.to_numeric(prime_video["titles_watched"], errors="coerce")
prime_video["completion_rate"] = pd.to_numeric(prime_video["completion_rate"], errors="coerce")

payments["payment_amount"] = pd.to_numeric(payments["payment_amount"], errors="coerce")
payments["retry_count"] = pd.to_numeric(payments["retry_count"], errors="coerce")

support["resolution_hours"] = pd.to_numeric(support["resolution_hours"], errors="coerce")
support["satisfaction_score"] = pd.to_numeric(support["satisfaction_score"], errors="coerce")

# Fill simple missing text values
customers[["state", "preferred_language", "acquisition_channel", "primary_device"]] = (
    customers[["state", "preferred_language", "acquisition_channel", "primary_device"]].fillna("Unknown")
)

memberships["discount_applied"] = memberships["discount_applied"].fillna("No Discount")

ended_status = memberships["membership_status"].isin(["Cancelled", "Expired"])
memberships.loc[ended_status & memberships["cancellation_reason"].isna(), "cancellation_reason"] = "Unknown"
memberships.loc[~ended_status & memberships["cancellation_reason"].isna(), "cancellation_reason"] = "Not Applicable"

orders[["product_category", "delivery_speed"]] = orders[["product_category", "delivery_speed"]].fillna("Unknown")
orders["shipping_fee_saved"] = orders["shipping_fee_saved"].fillna(orders["shipping_fee_saved"].median())

prime_video[["genre", "device_type"]] = prime_video[["genre", "device_type"]].fillna("Unknown")

payments["payment_method"] = payments["payment_method"].fillna("Unknown")
payments.loc[payments["payment_status"].eq("Failed") & payments["failure_reason"].isna(), "failure_reason"] = "Unknown"
payments.loc[~payments["payment_status"].eq("Failed") & payments["failure_reason"].isna(), "failure_reason"] = "Not Applicable"

support[["support_channel", "priority"]] = support[["support_channel", "priority"]].fillna("Unknown")

print("Data types and missing values handled")

Data types and missing values handled


In [10]:
# Removing records with missing important IDs or unrecoverable event dates
customers = customers.dropna(subset=["customer_id", "signup_date"])
memberships = memberships.dropna(subset=["membership_id", "customer_id", "membership_start_date"])
orders = orders.dropna(subset=["order_id", "customer_id", "order_date"])
prime_video = prime_video.dropna(subset=["activity_id", "customer_id", "activity_date"])
payments = payments.dropna(subset=["payment_id", "membership_id", "customer_id", "payment_date"])
support = support.dropna(subset=["ticket_id", "customer_id", "ticket_date"])

# Remove impossible membership and numeric records
memberships = memberships[
    (memberships["membership_end_date"].isna() | (memberships["membership_end_date"] >= memberships["membership_start_date"])) &
    (memberships["cancellation_date"].isna() | (memberships["cancellation_date"] >= memberships["membership_start_date"])) &
    (memberships["membership_fee"] >= 0)
]

orders = orders[(orders["order_value"] >= 0) & (orders["items_count"] > 0)]

prime_video = prime_video[
    (prime_video["watch_minutes"] >= 0) &
    (prime_video["sessions_count"] > 0) &
    (prime_video["titles_watched"] > 0)
]

prime_video = prime_video[
    prime_video["completion_rate"].isna() |
    prime_video["completion_rate"].between(0, 1)
]

payments = payments[(payments["payment_amount"] >= 0) & (payments["retry_count"] >= 0)]

support.loc[
    support["satisfaction_score"].notna() & ~support["satisfaction_score"].between(1, 5),
    "satisfaction_score"
] = np.nan

support.loc[
    support["resolution_hours"].notna() & (support["resolution_hours"] < 0),
    "resolution_hours"
] = np.nan

# Removing orphan records
memberships = memberships[memberships["customer_id"].isin(customers["customer_id"])]
orders = orders[orders["customer_id"].isin(customers["customer_id"])]
prime_video = prime_video[prime_video["customer_id"].isin(customers["customer_id"])]
payments = payments[payments["customer_id"].isin(customers["customer_id"])]
support = support[support["customer_id"].isin(customers["customer_id"])]
payments = payments[payments["membership_id"].isin(memberships["membership_id"])]

membership_owner = memberships.set_index("membership_id")["customer_id"]
payments = payments[payments["customer_id"].eq(payments["membership_id"].map(membership_owner))]

# Remove records before customer signup
signup_lookup = customers.set_index("customer_id")["signup_date"]
orders = orders[orders["order_date"] >= orders["customer_id"].map(signup_lookup)]
prime_video = prime_video[prime_video["activity_date"] >= prime_video["customer_id"].map(signup_lookup)]
payments = payments[payments["payment_date"] >= payments["customer_id"].map(signup_lookup)]
support = support[support["ticket_date"] >= support["customer_id"].map(signup_lookup)]

# Remove orders and video activity after the latest cancellation
latest_membership = memberships.sort_values(["customer_id", "membership_start_date"]).drop_duplicates("customer_id", keep="last")
latest_cancelled = latest_membership[latest_membership["membership_status"].eq("Cancelled")].set_index("customer_id")["cancellation_date"]

order_cancel_date = orders["customer_id"].map(latest_cancelled)
video_cancel_date = prime_video["customer_id"].map(latest_cancelled)

orders = orders[order_cancel_date.isna() | (orders["order_date"] <= order_cancel_date)]
prime_video = prime_video[video_cancel_date.isna() | (prime_video["activity_date"] <= video_cancel_date)]

print("Impossible and orphan records removed")

Impossible and orphan records removed


In [11]:
# Final validation
cleaned_dataframes = {
    "customers": customers,
    "memberships": memberships,
    "orders": orders,
    "prime_video_activity": prime_video,
    "payments": payments,
    "support_interactions": support
}

final_summary = []

for name, df in cleaned_dataframes.items():
    key = primary_keys[name]
    final_summary.append({
        "file": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "duplicate_primary_keys": df[key].duplicated().sum(),
        "missing_primary_keys": df[key].isna().sum()
    })

final_summary = pd.DataFrame(final_summary)
print(final_summary)

print("\nFinal orphan checks")
print("memberships:", (~memberships["customer_id"].isin(customers["customer_id"])).sum())
print("orders:", (~orders["customer_id"].isin(customers["customer_id"])).sum())
print("prime video:", (~prime_video["customer_id"].isin(customers["customer_id"])).sum())
print("payments customer:", (~payments["customer_id"].isin(customers["customer_id"])).sum())
print("payments membership:", (~payments["membership_id"].isin(memberships["membership_id"])).sum())
print("support:", (~support["customer_id"].isin(customers["customer_id"])).sum())

print("\nFinal status checks")
print("Invalid membership status:", (~memberships["membership_status"].isin(["Active", "Cancelled", "Expired", "Paused", "Payment Pending"])).sum())
print("Invalid payment status:", (~payments["payment_status"].isin(["Successful", "Failed", "Pending", "Refunded"])).sum())
print("Invalid satisfaction score:", (support["satisfaction_score"].notna() & ~support["satisfaction_score"].between(1, 5)).sum())

                   file    rows  columns  duplicate_rows  \
0             customers   50000       11               0   
1           memberships   55000       13               0   
2                orders  399976       11               0   
3  prime_video_activity  549970       10               0   
4              payments  124984       10               0   
5  support_interactions   34990       10               0   

   duplicate_primary_keys  missing_primary_keys  
0                       0                     0  
1                       0                     0  
2                       0                     0  
3                       0                     0  
4                       0                     0  
5                       0                     0  

Final orphan checks
memberships: 0
orders: 0
prime video: 0
payments customer: 0
payments membership: 0
support: 0

Final status checks
Invalid membership status: 0
Invalid payment status: 0
Invalid satisfaction score: 0


In [12]:
# Saving cleaned files
customers.to_csv(processed_path / "customers.csv", index=False, date_format="%Y-%m-%d")
memberships.to_csv(processed_path / "memberships.csv", index=False, date_format="%Y-%m-%d")
orders.to_csv(processed_path / "orders.csv", index=False, date_format="%Y-%m-%d")
prime_video.to_csv(processed_path / "prime_video_activity.csv", index=False, date_format="%Y-%m-%d")
payments.to_csv(processed_path / "payments.csv", index=False, date_format="%Y-%m-%d")
support.to_csv(processed_path / "support_interactions.csv", index=False, date_format="%Y-%m-%d")

print("Cleaned files saved in:", processed_path)
print(sorted([file.name for file in processed_path.glob("*.csv")]))

Cleaned files saved in: c:\Users\naren\OneDrive\Desktop\Data Analytics\Projects\Portfolio Centerpiece Projects using AI\End-to-End Amazon Prime Customer Churn & Retention Analytics\data\processed
['customers.csv', 'memberships.csv', 'orders.csv', 'payments.csv', 'prime_video_activity.csv', 'support_interactions.csv']
